# Pré-processamento do Acre - cópia com exportação numérica para aplicar os algoritimos de classificação
Cópia de `pre_processamento_acre.ipynb`, com as análises originais e uma nova seção de transformação ao final. A imputação de notas pela mediana foi removida desta cópia.

A exportação parte de `df_removidos`, criado logo após a leitura, com as cinco notas disponíveis. Assim, os filtros exploratórios posteriores (redação maior que zero, escolas específicas e outliers) não alteram a população exportada. As análises anteriores são exploratórias; seus rótulos agrupados e escalas não entram no CSV.

Configuração inicial

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None) # Limite de colunas que serão mostradas
pd.set_option("display.max_rows", 1000) # Limite de linhas que serão mostradas

plt.rcParams["figure.figsize"] = (10,6) # Tamanho das figuras, largura e altura


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/local/lib/python3.10/dist-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.10/dist-packages/traitlets/config/application.py", line 1053, in launch_instance
    app.start()
  File "/usr/local/lib/python3.10/dist-packages/ipykernel/kernelapp.p

AttributeError: _ARRAY_API not found

Importação da base

In [2]:
df_acre = pd.read_csv("base_dados/MICRODADOS_ENEM_2023_AC.csv")
print("Base carregada com sucesso.")

Base carregada com sucesso.


Checagem dos valores nulos

In [3]:
df_nulos = df_acre.isnull().sum().sort_values(ascending=False) # Contagem de nulos e ordenação decrescente
df_nulos = df_nulos.to_frame("Quantidade de nulos")
df_nulos["Porcentagem"] = (df_acre.isnull().sum() / len(df_acre) * 100).sort_values(ascending=False) # Coluna de porcentagem

print(df_nulos)

                        Quantidade de nulos  Porcentagem
SG_UF_ESC                             20077    82.709895
NO_MUNICIPIO_ESC                      20077    82.709895
TP_SIT_FUNC_ESC                       20077    82.709895
TP_DEPENDENCIA_ADM_ESC                20077    82.709895
TP_LOCALIZACAO_ESC                    20077    82.709895
CO_MUNICIPIO_ESC                      20077    82.709895
CO_UF_ESC                             20077    82.709895
TP_ENSINO                             17746    73.107028
NU_NOTA_CN                             9120    37.571064
NU_NOTA_MT                             9120    37.571064
CO_PROVA_MT                            9120    37.571064
CO_PROVA_CN                            9120    37.571064
NU_NOTA_COMP3                          8112    33.418472
CO_PROVA_CH                            8112    33.418472
NU_NOTA_CH                             8112    33.418472
NU_NOTA_LC                             8112    33.418472
CO_PROVA_LC                    

In [4]:
colunas_notas = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']

df_removidos = df_acre.dropna(subset=colunas_notas).copy() # Retira linhas que possuem algum valor de nota nulo

print(df_acre.shape)
print(df_removidos.shape)

df_removidos.head(10)

(24274, 65)
(15099, 65)


,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ESCOLA,TP_ENSINO,IN_TREINEIRO,CO_MUNICIPIO_ESC,NO_MUNICIPIO_ESC,CO_UF_ESC,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,CO_MUNICIPIO_PROVA,CO_UF_PROVA,SG_UF_PROVA,TP_PRESENCA_CN,TP_PRESENCA_CH,TP_PRESENCA_LC,TP_PRESENCA_MT,CO_PROVA_CN,CO_PROVA_CH,CO_PROVA_LC,CO_PROVA_MT,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,TP_LINGUA,TP_STATUS_REDACAO,NU_NOTA_COMP1,NU_NOTA_COMP2,NU_NOTA_COMP3,NU_NOTA_COMP4,NU_NOTA_COMP5,NU_NOTA_REDACAO,Q001,Q002,Q003,Q004,Q005,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023,Q024,Q025
0,6,F,0,3,1,1,0,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,1,1,1,1,1223.0,1194.0,1203.0,1213.0,369.9,387.1,438.3,360.2,1,1.0,100.0,120.0,100.0,80.0,0.0,400.0,E,C,D,B,4,C,A,B,D,A,C,B,A,A,B,B,A,A,B,A,A,E,A,B,B
4,6,F,1,3,2,1,1,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,1,1,1,1,1222.0,1191.0,1201.0,1212.0,331.8,476.9,454.5,342.6,0,1.0,80.0,80.0,80.0,80.0,40.0,360.0,H,B,C,B,2,B,A,B,D,A,A,B,A,B,A,A,A,A,B,A,A,C,A,A,B
5,8,F,1,2,1,1,7,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,1,1,1,1,1224.0,1192.0,1202.0,1214.0,596.2,691.9,647.1,669.0,1,1.0,120.0,200.0,120.0,120.0,80.0,640.0,E,E,B,B,5,C,A,B,C,A,A,B,A,A,A,A,A,A,B,A,A,E,A,B,B
6,3,F,1,1,1,2,0,2,1.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,1,1,1,1,1222.0,1191.0,1201.0,1212.0,444.4,369.4,446.7,497.9,1,1.0,120.0,60.0,80.0,120.0,40.0,420.0,D,D,C,B,3,A,A,B,A,A,B,B,B,A,B,A,A,A,B,A,A,D,A,A,B
7,3,F,1,3,1,2,0,2,1.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,1,1,1,1,1224.0,1192.0,1202.0,1214.0,467.5,540.3,576.7,536.2,1,1.0,120.0,120.0,120.0,140.0,120.0,620.0,C,B,C,A,4,B,A,A,D,A,A,B,B,B,A,A,A,A,B,A,A,D,A,A,B
8,4,F,1,3,1,1,1,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,1,1,1,1,1222.0,1191.0,1201.0,1212.0,402.0,471.4,521.9,411.7,1,1.0,120.0,40.0,40.0,100.0,40.0,340.0,D,B,C,B,3,B,A,B,C,B,A,B,A,A,A,A,A,A,B,A,A,D,A,A,A
10,13,F,1,3,1,1,17,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200203,12,AC,1,1,1,1,1223.0,1194.0,1203.0,1213.0,523.9,414.4,408.4,409.4,1,1.0,120.0,120.0,120.0,120.0,100.0,580.0,C,B,D,B,7,H,A,C,D,A,B,B,A,B,A,A,A,A,C,A,A,D,A,A,B
12,3,F,1,3,1,1,1,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,1,1,1,1,1223.0,1194.0,1203.0,1213.0,442.7,561.9,539.8,413.4,0,1.0,160.0,200.0,160.0,180.0,120.0,820.0,D,E,C,B,3,B,A,B,C,A,A,B,A,B,A,A,A,A,B,A,A,C,B,A,B
13,8,F,0,3,1,1,6,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,1,1,1,1,1223.0,1194.0,1203.0,1213.0,529.2,602.0,601.8,625.3,1,1.0,120.0,120.0,120.0,120.0,120.0,600.0,E,E,C,B,2,D,A,B,C,A,A,B,A,A,A,A,A,A,B,A,A,C,A,A,B
15,3,M,1,3,1,1,1,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200336,12,AC,1,1,1,1,1223.0,1194.0,1203.0,1213.0,543.1,478.9,522.6,494.5,1,1.0,140.0,200.0,140.0,160.0,200.0,840.0,D,C,B,B,6,B,A,B,C,A,A,B,B,B,A,A,A,A,B,A,A,B,A,A,A


# Transformação categórica e exportação sem imputação
Código direto, sem funções nem pipelines. Selecionamos os 33 atributos demográficos, escolares e socioeconômicos usados no projeto. Notas, códigos de prova e identificadores não entram nos preditores.

`media_geral` é mantida para conferência e `classe` é o alvo: é o alvo: Baixa (< 500), Média (500 a 700 inclusive) e Alta (> 700). Ambas devem ficar fora de X.

Escolaridade usa A=0 até G=6; H (Não sei) fica como ausente no nível escolar, com indicador próprio. Renda usa A=0 até Q=16. As demais categorias são codificadas conservadoramente por one-hot, sem supor distâncias entre categorias. Q005 continua numérica.

As categorias são declaradas previamente, sem aprender a lista com a base inteira. Nenhum valor ausente será substituído por moda, mediana ou zero. Nas dummies, todas as colunas referentes a um campo ausente continuam ausentes. O CSV pode, portanto, conter campos vazios e não é uma matriz completa pronta para KNN.

In [5]:
 # Seleciona os atributos que serão úteis para a o treinamento dos algoritimos de classificação
atributos = [
    "TP_FAIXA_ETARIA", "TP_SEXO", "TP_ESTADO_CIVIL", "TP_COR_RACA",
    "TP_ST_CONCLUSAO", "TP_ANO_CONCLUIU", "TP_ESCOLA", "TP_ENSINO",
] + [f"Q{i:03d}" for i in range(1, 26)]

df_numerico = df_removidos[atributos].copy()
media_geral = df_removidos[colunas_notas].mean(axis=1, skipna=False)
assert media_geral.notna().all()

In [6]:
# Mapeamentos fixos: não calculam estatísticas da base.
ordem_escolaridade = {letra: numero for numero, letra in enumerate("ABCDEFG")}
ordem_renda = {letra: numero for numero, letra in enumerate("ABCDEFGHIJKLMNOPQ")}

# Trata "Não sei" (H) como indicador numérico e preserva respostas em branco (NaN)
for coluna in ["Q001", "Q002"]:
    assert df_numerico[coluna].dropna().isin(list("ABCDEFGH")).all()
    indicador = df_numerico[coluna].eq("H").astype(float)
    indicador.loc[df_numerico[coluna].isna()] = np.nan
    df_numerico[f"{coluna}_nao_sei"] = indicador
    df_numerico[coluna] = df_numerico[coluna].map(ordem_escolaridade)

assert df_numerico["Q006"].dropna().isin(list(ordem_renda)).all()
df_numerico["Q006"] = df_numerico["Q006"].map(ordem_renda)
# Garante que a quantidade de moradores (Q005) seja númerica
df_numerico["Q005"] = pd.to_numeric(df_numerico["Q005"], errors="raise")

In [7]:
# Categorias declaradas explicitamente, inclusive as já representadas por números.
categorias = {
    "TP_FAIXA_ETARIA": list(range(1, 21)),
    "TP_SEXO": ["F", "M"],
    "TP_ESTADO_CIVIL": list(range(5)),
    "TP_COR_RACA": list(range(7)),
    "TP_ST_CONCLUSAO": [1, 2, 3, 4],
    "TP_ANO_CONCLUIU": list(range(18)),
    "TP_ESCOLA": [1, 2, 3],
    "TP_ENSINO": [1, 2, 3],
    "Q003": list("ABCDEF"),
    "Q004": list("ABCDEF"),
    "Q007": list("ABCD"),
}
for coluna in ["Q008", "Q009", "Q010", "Q011", "Q012", "Q013", "Q014", "Q015", "Q016", "Q017", "Q019", "Q022", "Q024"]:
    categorias[coluna] = list("ABCDE")
for coluna in ["Q018", "Q020", "Q021", "Q023", "Q025"]:
    categorias[coluna] = list("AB")

for coluna, valores in categorias.items():
    assert df_numerico[coluna].dropna().isin(valores).all()
    ausentes = df_numerico[coluna].isna()
    serie = pd.Series(
        pd.Categorical(df_numerico[coluna], categories=valores),
        index=df_numerico.index,
        name=coluna,
    )
    dummies = pd.get_dummies(
        serie, prefix=coluna, dtype=float,
        drop_first=(len(valores) == 2),
    )
    # Preserva ausentes: não confundir dado desconhecido com a categoria de referência.
    dummies.loc[ausentes, :] = np.nan
    df_numerico = pd.concat([df_numerico.drop(columns=coluna), dummies], axis=1)

In [8]:
# Média e alvo numérico, sem arredondar antes de classificar.
df_numerico["media_geral"] = media_geral
df_numerico["classe"] = np.select(
    [media_geral < 500, media_geral <= 700], ["Baixa", "Media"], default="Alta",
).astype("str")

# Restaura a ordem dos atributos da base original e mantém o questionário socioeconômico no final.
# Colunas derivadas (indicadores e one-hot) permanecem junto da variável que as originou.
colunas_ordenadas = []
for atributo in atributos:
    if atributo in df_numerico.columns:
        colunas_ordenadas.append(atributo)
    colunas_ordenadas.extend(
        coluna for coluna in df_numerico.columns
        if coluna.startswith(f"{atributo}_")
    )

df_numerico = df_numerico[colunas_ordenadas + ["media_geral", "classe"]]
df_numerico = df_numerico.reset_index(drop=True)
df_numerico.head(10)

,TP_FAIXA_ETARIA_1,TP_FAIXA_ETARIA_2,TP_FAIXA_ETARIA_3,TP_FAIXA_ETARIA_4,TP_FAIXA_ETARIA_5,TP_FAIXA_ETARIA_6,TP_FAIXA_ETARIA_7,TP_FAIXA_ETARIA_8,TP_FAIXA_ETARIA_9,TP_FAIXA_ETARIA_10,TP_FAIXA_ETARIA_11,TP_FAIXA_ETARIA_12,TP_FAIXA_ETARIA_13,TP_FAIXA_ETARIA_14,TP_FAIXA_ETARIA_15,TP_FAIXA_ETARIA_16,TP_FAIXA_ETARIA_17,TP_FAIXA_ETARIA_18,TP_FAIXA_ETARIA_19,TP_FAIXA_ETARIA_20,TP_SEXO_M,TP_ESTADO_CIVIL_0,TP_ESTADO_CIVIL_1,TP_ESTADO_CIVIL_2,TP_ESTADO_CIVIL_3,TP_ESTADO_CIVIL_4,TP_COR_RACA_0,TP_COR_RACA_1,TP_COR_RACA_2,TP_COR_RACA_3,TP_COR_RACA_4,TP_COR_RACA_5,TP_COR_RACA_6,TP_ST_CONCLUSAO_1,TP_ST_CONCLUSAO_2,TP_ST_CONCLUSAO_3,TP_ST_CONCLUSAO_4,TP_ANO_CONCLUIU_0,TP_ANO_CONCLUIU_1,TP_ANO_CONCLUIU_2,TP_ANO_CONCLUIU_3,TP_ANO_CONCLUIU_4,TP_ANO_CONCLUIU_5,TP_ANO_CONCLUIU_6,TP_ANO_CONCLUIU_7,TP_ANO_CONCLUIU_8,TP_ANO_CONCLUIU_9,TP_ANO_CONCLUIU_10,TP_ANO_CONCLUIU_11,TP_ANO_CONCLUIU_12,TP_ANO_CONCLUIU_13,TP_ANO_CONCLUIU_14,TP_ANO_CONCLUIU_15,TP_ANO_CONCLUIU_16,TP_ANO_CONCLUIU_17,TP_ESCOLA_1,TP_ESCOLA_2,TP_ESCOLA_3,TP_ENSINO_1,TP_ENSINO_2,TP_ENSINO_3,Q001,Q001_nao_sei,Q002,Q002_nao_sei,Q003_A,Q003_B,Q003_C,Q003_D,Q003_E,Q003_F,Q004_A,Q004_B,Q004_C,Q004_D,Q004_E,Q004_F,Q005,Q006,Q007_A,Q007_B,Q007_C,Q007_D,Q008_A,Q008_B,Q008_C,Q008_D,Q008_E,Q009_A,Q009_B,Q009_C,Q009_D,Q009_E,Q010_A,Q010_B,Q010_C,Q010_D,Q010_E,Q011_A,Q011_B,Q011_C,Q011_D,Q011_E,Q012_A,Q012_B,Q012_C,Q012_D,Q012_E,Q013_A,Q013_B,Q013_C,Q013_D,Q013_E,Q014_A,Q014_B,Q014_C,Q014_D,Q014_E,Q015_A,Q015_B,Q015_C,Q015_D,Q015_E,Q016_A,Q016_B,Q016_C,Q016_D,Q016_E,Q017_A,Q017_B,Q017_C,Q017_D,Q017_E,Q018_B,Q019_A,Q019_B,Q019_C,Q019_D,Q019_E,Q020_B,Q021_B,Q022_A,Q022_B,Q022_C,Q022_D,Q022_E,Q023_B,Q024_A,Q024_B,Q024_C,Q024_D,Q024_E,Q025_B,media_geral,classe
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,4.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,4,2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,391.10,Baixa
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2,1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,393.16,Baixa
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,4.0,0.0,4.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,5,2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,648.84,Media
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,3.0,0.0,3.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,

In [9]:
caminho_saida = "base_dados/MICRODADOS_ENEM_2023_AC_PARA_REGRESSAO.csv"
df_numerico.to_csv(caminho_saida, index=False, encoding="utf-8")

# Conferência do arquivo exportado.
conferencia = pd.read_csv(caminho_saida)
assert conferencia.shape == df_numerico.shape
assert conferencia.isna().equals(df_numerico.isna())
assert conferencia["classe"].equals(df_numerico["classe"].astype(conferencia["classe"].dtype))
print(f"CSV exportado: {caminho_saida}")
print(f"Dimensões: {df_numerico.shape[0]} linhas e {df_numerico.shape[1]} colunas")

CSV exportado: base_dados/MICRODADOS_ENEM_2023_AC_PARA_REGRESSAO.csv
Dimensões: 15099 linhas e 155 colunas
